In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the CSV
df = pd.read_csv('detailed_estimates.csv')

# Remove specified columns
cols_to_drop = ['run_id', 'phase', 'technique_ids', "failure_mode_id", 'round_num', 'rationale', 'key_assumptions', "error"]
df = df.drop(columns=cols_to_drop)

# Reset index
df = df.reset_index(drop=True)

# Save to a new CSV
df.to_csv('cleaned_outputs.csv', index=False)

In [9]:
import pandas as pd
import os
import re

# Load the cleaned CSV
df = pd.read_csv('cleaned_outputs.csv')

# Function to sanitize filenames
def sanitize_filename(name):
    # Replace spaces with underscores
    name = name.replace(' ', '_')
    # Remove invalid characters
    name = re.sub(r'[<>:"/\\|?*]', '', name)
    # Remove trailing underscores
    name = name.rstrip('_')
    return name

# Define clustering configs:
# (folder_name, groupby_col, drop_col, sort_within_group_col)
configs = [
    ('by_expert',        'expert',            'expert',            'technique_names'),
    ('by_technique',     'technique_names',   'technique_names',   'failure_mode_name'),
    ('by_failure_mode',  'failure_mode_name', 'failure_mode_name', 'expert'),
]

for folder, group_col, drop_col, sort_col in configs:
    os.makedirs(folder, exist_ok=True)  # create the top-level folder

    for name, group in df.groupby(group_col):
        sorted_group = (group
                        .sort_values([sort_col, 'estimate'])
                        .drop(columns=[drop_col])
                        .reset_index(drop=True))

        filename = f"{sanitize_filename(str(name))}.csv"
        filepath = os.path.join(folder, filename)
        sorted_group.to_csv(filepath, index=False)

In [10]:
# Same configs as before
configs = [
    ('by_expert',        'technique_names'),
    ('by_technique',     'failure_mode_name'),
    ('by_failure_mode',  'expert'),
]

for folder, x_col in configs:
    graphs_folder = os.path.join(folder, 'Mean_Graphs')
    os.makedirs(graphs_folder, exist_ok=True)

    for filename in os.listdir(folder):
        if not filename.endswith('.csv'):
            continue

        filepath = os.path.join(folder, filename)
        df = pd.read_csv(filepath)

        if df.empty or x_col not in df.columns:
            continue

        # Aggregate mean estimate per x_col group
        grouped = df.groupby(x_col)['estimate'].mean().reset_index()

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.bar(grouped[x_col], grouped['estimate'], color='steelblue', edgecolor='black')

        ax.set_xlabel(x_col.replace('_', ' ').title(), fontsize=12)
        ax.set_ylabel('Mean Estimate', fontsize=12)
        ax.set_title(filename.replace('.csv', '').replace('_', ' '), fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=9)
        plt.tight_layout()

        graph_filename = filename.replace('.csv', '.png')
        plt.savefig(os.path.join(graphs_folder, graph_filename), dpi=150)
        plt.close()

In [11]:
# folder, x_col, label_col
configs = [
    ('by_expert',        'technique_names',   'failure_mode_name'),
    ('by_technique',     'failure_mode_name',  'expert'),
    ('by_failure_mode',  'expert',             'technique_names'),
]

for folder, x_col, label_col in configs:
    graphs_folder = os.path.join(folder, 'MinMax_Graphs')
    os.makedirs(graphs_folder, exist_ok=True)

    for filename in os.listdir(folder):
        if not filename.endswith('.csv'):
            continue

        filepath = os.path.join(folder, filename)
        df = pd.read_csv(filepath)

        if df.empty or x_col not in df.columns:
            continue

        # For each x_col group, get the label_col value at min and max estimate
        grouped = df.groupby(x_col).apply(
            lambda g: pd.Series({
                'min_val':   g.loc[g['estimate'].idxmin(), 'estimate'],
                'max_val':   g.loc[g['estimate'].idxmax(), 'estimate'],
                'min_label': g.loc[g['estimate'].idxmin(), label_col],
                'max_label': g.loc[g['estimate'].idxmax(), label_col],
            })
        ).reset_index()

        x = np.arange(len(grouped))
        width = 0.3

        fig, ax = plt.subplots(figsize=(14, 7))

        bars_min = ax.bar(x - width/2, grouped['min_val'], width=width, color='steelblue', edgecolor='black', label='Minimum')
        bars_max = ax.bar(x + width/2, grouped['max_val'], width=width, color='tomato',    edgecolor='black', label='Maximum')

        # Max bars: write label_col name INSIDE the bar
        for bar, label in zip(bars_max, grouped['max_label']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                    str(label), ha='center', va='center',
                    fontsize=6.5, color='white', fontweight='bold',
                    rotation=90)

        # Min bars: write label_col name ABOVE the bar
        for bar, label in zip(bars_min, grouped['min_label']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(label), ha='center', va='bottom',
                    fontsize=6.5, color='steelblue', fontweight='bold',
                    rotation=90)

        ax.set_xticks(x)
        ax.set_xticklabels(grouped[x_col], rotation=45, ha='right', fontsize=9)
        ax.set_xlabel(x_col.replace('_', ' ').title(), fontsize=12)
        ax.set_ylabel('Estimate', fontsize=12)
        ax.set_title(filename.replace('.csv', '').replace('_', ' '), fontsize=14)
        ax.legend()
        plt.tight_layout()

        graph_filename = filename.replace('.csv', '.png')
        plt.savefig(os.path.join(graphs_folder, graph_filename), dpi=150)
        plt.close()

In [12]:
# folder, x_col, label_col
configs = [
    ('by_expert',        'technique_names',   'failure_mode_name'),
    ('by_technique',     'failure_mode_name',  'expert'),
    ('by_failure_mode',  'expert',             'technique_names'),
]

for folder, x_col, label_col in configs:
    graphs_folder = os.path.join(folder, 'Graphs')
    os.makedirs(graphs_folder, exist_ok=True)

    for filename in os.listdir(folder):
        if not filename.endswith('.csv'):
            continue

        filepath = os.path.join(folder, filename)
        df = pd.read_csv(filepath)

        if df.empty or x_col not in df.columns:
            continue

        # Aggregate min, max, mean per x_col group
        grouped = df.groupby(x_col).apply(
            lambda g: pd.Series({
                'min_val':   g.loc[g['estimate'].idxmin(), 'estimate'],
                'max_val':   g.loc[g['estimate'].idxmax(), 'estimate'],
                'mean_val':  g['estimate'].mean(),
                'min_label': g.loc[g['estimate'].idxmin(), label_col],
                'max_label': g.loc[g['estimate'].idxmax(), label_col],
            })
        ).reset_index()

        x = np.arange(len(grouped))
        width = 0.25

        fig, ax = plt.subplots(figsize=(14, 7))

        bars_min  = ax.bar(x - width,     grouped['min_val'],  width=width, color='steelblue', edgecolor='black', label='Minimum')
        bars_mean = ax.bar(x,             grouped['mean_val'], width=width, color='mediumseagreen', edgecolor='black', label='Mean')
        bars_max  = ax.bar(x + width,     grouped['max_val'],  width=width, color='tomato',    edgecolor='black', label='Maximum')

        # Max bars: label INSIDE the bar
        for bar, label in zip(bars_max, grouped['max_label']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                    str(label), ha='center', va='center',
                    fontsize=6.5, color='white', fontweight='bold',
                    rotation=90)

        # Min bars: label ABOVE the bar
        for bar, label in zip(bars_min, grouped['min_label']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(label), ha='center', va='bottom',
                    fontsize=6.5, color='steelblue', fontweight='bold',
                    rotation=90)

        # Mean bars: write the mean value inside the bar
        for bar, val in zip(bars_mean, grouped['mean_val']):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                    f'{val:.1f}', ha='center', va='center',
                    fontsize=6.5, color='white', fontweight='bold',
                    rotation=90)

        ax.set_xticks(x)
        ax.set_xticklabels(grouped[x_col], rotation=45, ha='right', fontsize=9)
        ax.set_xlabel(x_col.replace('_', ' ').title(), fontsize=12)
        ax.set_ylabel('Estimate', fontsize=12)
        ax.set_title(filename.replace('.csv', '').replace('_', ' '), fontsize=14)
        ax.legend()
        plt.tight_layout()

        graph_filename = filename.replace('.csv', '.png')
        plt.savefig(os.path.join(graphs_folder, graph_filename), dpi=150)
        plt.close()